In [1]:
# command to reload libraries
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
from desReg.utils import instance_hardness as ih

pd.set_option('display.max_columns', None)

/Users/victor.nunes/Documents/Github/regression-instance-hardness/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [3]:
from ensemble_versions import des_baseline_ensemble, des_challenger_experiments, des_ensemble_s2, se_ensemble_l1
from run_experiments import run_experiment_with_timeout
from desReg.des.DESRegression import DESRegression
from desReg.utils import measures as em
from sklearn import linear_model
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor

regressors_list = [
    linear_model.Lasso(), 
    linear_model.Ridge(), 
    KNeighborsRegressor(), 
    SVR(), 
    DecisionTreeRegressor()
]

des_ensemble_ih = DESRegression(
    regressors_list = regressors_list,
    random_state=0,
    n_estimators_bag=20,
    competence_region='knn',
    k=5,
    competence_level=em.sum_absolute_error_weighted,
    include_instance_hardness=True,
    include_hardness_measures=False,
    hardness_measures_list=None,
    meta_feature='instance_hardness',
    ensemble_type='DES'
)

drs_ensemble_ih = DESRegression(
    regressors_list = regressors_list,
    random_state=0,
    n_estimators_bag=20,
    competence_region='knn',
    k=5,
    competence_level=em.sum_absolute_error_weighted,
    include_instance_hardness=True,
    include_hardness_measures=False,
    hardness_measures_list=None,
    meta_feature='instance_hardness',
    ensemble_type='DRS'
)

des_all_errors_weighted = DESRegression(
    regressors_list = regressors_list,
    random_state=0,
    n_estimators_bag=20,
    competence_region='knn',
    k=5,
    competence_level=em.all_errors_weighted,
    include_instance_hardness=True,
    include_hardness_measures=False,
    hardness_measures_list=None,
    meta_feature='instance_hardness',
    ensemble_type='DES'
)

des_inverse_ih_all_errors_weighted = DESRegression(
    regressors_list = regressors_list,
    random_state=0,
    n_estimators_bag=20,
    competence_region='knn',
    k=5,
    competence_level=em.all_errors_weighted,
    include_instance_hardness=True,
    include_hardness_measures=False,
    hardness_measures_list=None,
    meta_feature='inverse_instance_hardness',
    ensemble_type='DES'
)

challenger_experiments = {
    "des_ensemble_ih": {"model": des_ensemble_ih},
    # 'drs_ensemble_ih': {"model": drs_ensemble_ih},
    # 'des_ensemble_s2': {"model": des_ensemble_s2},
    # 'se_ensemble_l1': {"model": se_ensemble_l1},
    # 'des_all_errors_weighted': {"model": des_all_errors_weighted},
    # 'des_inverse_ih_all_errors_weighted': {"model": des_inverse_ih_all_errors_weighted},
}


In [24]:
from run_experiments_refactored import run_experiment
from desReg.des.DESRegression import DESRegression
from desReg.utils import measures as em
from sklearn import linear_model
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor

regressors_list = [
    linear_model.Lasso(), 
    linear_model.Ridge(), 
    KNeighborsRegressor(), 
    SVR(), 
    DecisionTreeRegressor()
]

des_baseline_ensemble = DESRegression(
    regressors_list=regressors_list,
    random_state=0,
    n_estimators_bag=20,
    competence_region='knn',
    k=5,
    competence_level=em.all_errors,
    include_instance_hardness=False
)

des_ensemble_ih = DESRegression(
    regressors_list = regressors_list,
    random_state=0,
    n_estimators_bag=20,
    competence_region='knn',
    k=5,
    competence_level=em.sum_absolute_error_weighted,
    include_instance_hardness=True,
    include_hardness_measures=False,
    hardness_measures_list=None,
    meta_feature='instance_hardness',
    ensemble_type='DES'
)

challenger_experiments = {
    'ih_des': {
        'model': des_ensemble_ih
    }
}
results_refactored = run_experiment(
    dataset_alias='student_marks',
    baseline_model=des_baseline_ensemble,
    challenger_experiments=challenger_experiments
)


========== Processing student_marks - Fold 1 ==========
Running baseline model...


desReg.des.DESRegression - INFO - Predicting DES
desReg.des.DESRegression - INFO - Calculating instance hardness


Running challenger model: ih_des...


desReg.des.DESRegression - INFO - Predicting DES
desReg.des.DESRegression - INFO - Predicting DES


========== Processing student_marks - Fold 2 ==========
Running baseline model...


desReg.des.DESRegression - INFO - Calculating instance hardness


... (10 outputs left out)

========== Processing student_marks - Fold 5 ==========
Running baseline model...


desReg.des.DESRegression - INFO - Calculating instance hardness


Running challenger model: ih_des...


desReg.des.DESRegression - INFO - Predicting DES


In [25]:
from run_experiments_refactored import get_df_results, get_agg_results

des_student_marks_results_df = get_df_results(results_refactored)

In [26]:
des_student_marks_agg_results = get_agg_results(des_student_marks_results_df)
des_student_marks_agg_results

mse         execution_time                  partition_size  \
              mean     std           mean     std      sum            sum   
model_name                                                                  
baseline    2.3476  1.5707         3.5850  2.8112  17.9249            100   
ih_des      1.8081  1.3873         4.6653  0.6456  23.3263            100   

           ensemble_size_Lasso ensemble_size_Ridge  \
                          mean                mean   
model_name                                           
baseline                  6.55                6.83   
ih_des                    5.06                6.43   

           ensemble_size_KNeighborsRegressor  \
                                        mean   
model_name                                     
baseline                               16.74   
ih_des                                 16.73   

           ensemble_size_DecisionTreeRegressor ensemble_size_TotalRegressors  \
                                          mean                          mean   
model_name                                                                     
baseline                                 17.57                         60.19   
ih_des                                   19.10                         57.84   

           ensemble_size_DistinctRegressors ensemble_size_SVR  
                                       mean              mean  
model_name                                                     
baseline                               3.84             12.50  
ih_des                                 3.70             10.52

In [4]:

des_student_marks_results = run_experiment_with_timeout(
    'student_marks', 
    baseline_model=des_baseline_ensemble, 
    # challenger_experiments=des_challenger_experiments
    challenger_experiments=challenger_experiments
)

========== Processing student_marks - Fold 1 ==========
Running baseline model...


desReg.des.DESRegression - INFO - Calculating instance hardness


Running challenger model: des_ensemble_ih...
========== Processing student_marks - Fold 2 ==========
Running baseline model...


desReg.des.DESRegression - INFO - Calculating instance hardness


Running challenger model: des_ensemble_ih...
========== Processing student_marks - Fold 3 ==========
Running baseline model...


desReg.des.DESRegression - INFO - Calculating instance hardness


Running challenger model: des_ensemble_ih...
========== Processing student_marks - Fold 4 ==========
Running baseline model...


desReg.des.DESRegression - INFO - Calculating instance hardness


Running challenger model: des_ensemble_ih...
========== Processing student_marks - Fold 5 ==========
Running baseline model...


desReg.des.DESRegression - INFO - Calculating instance hardness


Running challenger model: des_ensemble_ih...


In [11]:
print(des_student_marks_results[1]['baseline']['ensemble_sizes'])
print(des_student_marks_results[2]['baseline']['ensemble_sizes'])
print(des_student_marks_results[3]['baseline']['ensemble_sizes'])
print(des_student_marks_results[4]['baseline']['ensemble_sizes'])
print(des_student_marks_results[5]['baseline']['y_pred'])

Counter({'TotalRegressors': 1216, 'DecisionTreeRegressor': 354, 'KNeighborsRegressor': 327, 'SVR': 249, 'Lasso': 145, 'Ridge': 141, 'DistinctRegressors': 79})
Counter({'TotalRegressors': 1148, 'DecisionTreeRegressor': 341, 'KNeighborsRegressor': 333, 'SVR': 248, 'Ridge': 119, 'Lasso': 107, 'DistinctRegressors': 76})
Counter({'TotalRegressors': 1217, 'DecisionTreeRegressor': 357, 'KNeighborsRegressor': 335, 'SVR': 225, 'Ridge': 165, 'Lasso': 135, 'DistinctRegressors': 79})
Counter({'TotalRegressors': 1186, 'DecisionTreeRegressor': 347, 'KNeighborsRegressor': 334, 'SVR': 279, 'Lasso': 125, 'Ridge': 101, 'DistinctRegressors': 75})
[20.660936754515156, 8.340487807946408, 14.705208094630862, 47.37182998755207, 50.30867379930911, 18.028537042815337, 30.297321510859568, 16.811122210703363, 20.962542827521197, 31.08035099252729, 39.52739148895547, 13.060815715450756, 24.310050852111104, 19.72312143209584, 12.308861612985698, 20.451215502761517, 32.03312137565915, 39.129659141982984, 46.5664873

In [16]:
print(len(des_student_marks_results[5]['baseline']['y_pred']))

20


In [18]:
from run_experiments import get_df_results, get_agg_results

des_student_marks_results_df = get_df_results(des_student_marks_results)
des_student_marks_results_df

,mse,r2,mae,rmse,mape,execution_time,y_pred,y_true,ensemble_sizes,status,model_name,partition,partition_size
0,1.734700,0.990517,0.872796,1.317080,0.047692,6.958291,"[50.472939323582224, 17.23408933572179, 41.181...","[54.321, 17.705, 44.099, 16.106, 16.461, 39.95...","{'Lasso': 145, 'Ridge': 141, 'KNeighborsRegres...",success,baseline,1,20
1,1.264335,0.993088,0.745771,1.124426,0.035255,1.751950,"[51.03057284739721, 16.90148900991777, 41.5819...","[54.321, 17.705, 44.099, 16.106, 16.461, 39.95...","{'Lasso': 116, 'Ridge': 132, 'KNeighborsRegres...",success,des_ensemble_ih,1,20
2,0.965786,0.993813,0.827634,0.982744,0.070903,0.928026,"[11.75769240849949, 11.964466567256318, 19.366...","[10.522, 10.844, 19.59, 21.379, 12.591, 13.562...","{'KNeighborsRegressor': 333, 'SVR': 248, 'Deci...",success,baseline,2,20
3,0.578182,0.996296,0.632546,0.760383,0.053755,1.610463,"[11.584835978524449, 12.36819294851366, 19.338...","[10.522, 10.844, 19.59, 21.379, 12.591, 13.562...","{'KNeighborsRegressor': 335, 'SVR': 172, 'Deci...",success,des_ensemble_ih,2,20
4,2.863547,0.988467,1.097354,1.692202,0.041018,0.924700,"[48.360788291782264, 8.581267252806894, 15.956...","[51.142, 7.336, 15.725, 19.771, 10.429, 9.742,...","{'Lasso': 135, 'Ridge': 165, 'KNeighborsRegres...",success,baseline,3,20
5,2.285403,0.990795,1.036478,1.511755,0.038031,1.608378,"[48.86310694424543, 8.267655606210903, 16.0546...","[51.142, 7.336, 15.725, 19.771, 10.429, 9.742,...","{'Lasso': 97, 'Ridge': 159, 'KNeighborsRegress...",success,des_ensemble_ih,3,20
6,1.323183,0.991633,0.856841,1.150297,0.043725,0.930329,"[23.72154957711868, 22.396666276899253, 37.265...","[25.133, 22.073, 35.939, 12.209, 28.043, 16.51...","{'KNeighborsRegressor': 334, 'SVR': 279, 'Deci...",success,baseline,4,20
7,0.912039,0.994233,0.747986,0.955007,0.041567,1.601602,"[23.660760403500486, 22.34467445719573, 36.269...","[25.133, 22.073, 35.939, 12.209, 28.043, 16.51...","{'KNeighborsRegressor': 328, 'SVR': 232, 'Deci...",success,des_ensemble_ih,4,20
8,4.851027,0.975090,1.516179,2.202505,0.053759,0.920254,"[20.660936754515156, 8.340487807946408, 14.705...","[19.202, 7.734, 13.811, 53.018, 55.299, 17.822...","{'Lasso': 143, 'Ridge': 157, 'KNeighborsRegres...",success,baseline,5,20
9,4.029326,0.979310,1.360539,2.007318,0.050952,1.600677,"[20.62316896974212, 6.868286500514223, 14.6726...","[19.202, 7.734, 13.811, 53.018, 55.299, 17.822...","{'KNeighborsRegressor': 340, 'SVR': 230, 'Deci...",success,des_ensemble_ih,5,20


In [19]:
des_student_marks_agg_results = get_agg_results(des_student_marks_results_df)
des_student_marks_agg_results

mse          \
                   mean     std   
model_name                        
baseline         2.3476  1.5707   
des_ensemble_ih  1.8139  1.3940   

                                                    ensemble_sizes  \
                                                         sum_freqs   
model_name                                                           
baseline         {'Lasso': 655, 'Ridge': 683, 'KNeighborsRegres...   
des_ensemble_ih  {'Lasso': 506, 'Ridge': 649, 'KNeighborsRegres...   

                execution_time                  partition_size algo_name  \
                          mean     std      sum            sum     Lasso   
model_name                                                                 
baseline                2.1323  2.6978  10.6616            100       655   
des_ensemble_ih         1.6346  0.0657   8.1731            100       506   

                                                                 \
                Ridge KNeighborsRegressor DecisionTreeRegressor   
model_name                                                        
baseline          683                1674                  1757   
des_ensemble_ih   649                1675                  1907   

                                                          
                TotalRegressors DistinctRegressors   SVR  
model_name                                                
baseline                   6019                384  1250  
des_ensemble_ih            5785                371  1048

In [ ]:
# Generate a random regression dataset with 1000 instances and 10 features using sklearn
from sklearn.datasets import make_regression

X, y = make_regression(n_samples=1000, n_features=10, noise=10, random_state=42)

hardness_measures = ih.get_hardness_measures(X, y)
hardness_measures.head()